# HW02 — MLflow Experiment Tracking

In HW01, you built a versioned feature dataset for the Airbnb listing availability problem.

In this notebook, you will train several model versions and track them in MLflow.

The goal is not only to get a high score. The goal is to make every experiment reproducible:

- which dataset version was used
- which features were used
- which model was trained
- which parameters were used
- which metrics were produced
- which artifacts were saved
- which run should be considered the final candidate

MLflow server:

```text
http://185.50.38.163:33014
```

Use your assigned MLflow username/password and your assigned experiment name from the credentials sheet.

## Required output

By the end of this notebook, you must have:

1. At least **5 MLflow runs**.
2. At least **3 different experiment types**:
   - one intentionally leaky run
   - one baseline run
   - at least one clean real model
3. Logged parameters, metrics, tags, artifacts, and an sklearn Pipeline model.
4. A run comparison table.
5. One selected final candidate run.
6. A short explanation of why that run was selected.

Do not use future/label columns in your final clean model.

In [1]:
# If needed, install these in your local environment first:
# pip install pandas numpy scikit-learn matplotlib mlflow pyarrow

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)

import mlflow
import mlflow.sklearn

RANDOM_STATE = 42

## 1. Configure MLflow

Fill in your assigned MLflow credentials.

Important:

- `MLFLOW_TRACKING_URI` is the shared MLflow server.
- `MLFLOW_USERNAME` and `MLFLOW_PASSWORD` are **not** your database credentials.
- `EXPERIMENT_NAME` must be your own assigned experiment name, for example:

```text
qbc12_hw02_student_nazanin_hesari
```

Do not use someone else's experiment name.

In [19]:
MLFLOW_TRACKING_URI = "http://185.50.38.163:33014"

# TODO: replace these with your assigned MLflow credentials.
MLFLOW_USERNAME = "student_atiyeh_attaran"
MLFLOW_PASSWORD = "_xp7o3mL4JCM76hNGXY"
EXPERIMENT_NAME = "qbc12_hw02_student_atiyeh_attaran"

if MLFLOW_USERNAME == "student_your_username" or MLFLOW_PASSWORD == "your_mlflow_password":
    raise ValueError("Replace MLFLOW_USERNAME, MLFLOW_PASSWORD, and EXPERIMENT_NAME with your assigned values.")

os.environ["MLFLOW_TRACKING_USERNAME"] = MLFLOW_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = MLFLOW_PASSWORD

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", experiment.name if experiment else None)
print("Experiment ID:", experiment.experiment_id if experiment else None)

MLflow tracking URI: http://185.50.38.163:33014
Experiment: qbc12_hw02_student_atiyeh_attaran
Experiment ID: 29


## 2. Load the HW01 dataset

Use the cleaned dataset produced by HW01.

Expected files:

```text
data/features/listing_availability_features_v1_audit_cleaned.csv
data/features/listing_availability_features_v1_audit_cleaned.parquet
data/features/listing_availability_features_v1_audit_cleaned_metadata.json
```

You may use CSV or Parquet. Parquet is preferred if available.

In [4]:
DATASET_VERSION = "v1_student"

FEATURE_DIR = Path("../HW02_A/data/features")

parquet_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.parquet"
csv_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}.csv"
metadata_path = FEATURE_DIR / f"listing_availability_features_{DATASET_VERSION}_metadata.json"

# load the dataset.
# Prefer Parquet if it exists, otherwise use CSV.
if parquet_path.exists():
    feature_df=pd.read_parquet(parquet_path)
else:
    feature_df=pd.read_csv(csv_path)
# load metadata if metadata_path exists.
metadata = {}
if metadata_path.exists():
    with open(metadata_path,"r") as f:
        metadata=json.load(f)
print(feature_df.shape)
feature_df.head()

(10480, 33)


,listing_id,future_calendar_days_observed_30d,future_available_days_30d,future_available_rate_30d,high_demand_proxy,property_type,room_type,accommodates,bathrooms,bedrooms,...,available_days_last_90d,available_rate_last_90d,avg_minimum_nights_calendar_last_90d,avg_maximum_nights_calendar_last_90d,available_days_last_30d,available_rate_last_30d,avg_minimum_nights_calendar_last_30d,avg_maximum_nights_calendar_last_30d,cutoff_date,dataset_version
0,1443670960781261954,30,30,1.0,0,Entire rental unit,Entire home/apt,2,1.0,1.0,...,91,1.000,2.0,30.0,31,1.0000,2.0,30.0,2026-08-11,v1_student
1,896043282611946316,30,0,0.0,1,Entire rental unit,Entire home/apt,2,1.5,1.0,...,0,0.000,5.0,25.0,0,0.0000,5.0,25.0,2026-08-11,v1_student
2,958726726744532841,30,0,0.0,1,Entire condo,Entire home/apt,2,1.0,1.0,...,0,0.000,2.0,7.0,0,0.0000,2.0,7.0,2026-08-11,v1_student
3,39969190,30,0,0.0,1,Entire rental unit,Entire home/apt,2,1.5,1.0,...,0,0.000,3.0,9.0,0,0.0000,3.0,9.0,2026-08-11,v1_student
4,1272264495001498383,30,30,1.0,0,Private room in townhouse,Private room,2,2.0,1.0,...,88,0.967,2.0,365.0,28,0.9032,2.0,365.0,2026-08-11,v1_student


## 3. Define target and forbidden columns

The target is:

```text
high_demand_proxy
```

The following columns must **not** be used as clean model inputs:

```text
listing_id
cutoff_date
dataset_version
future_calendar_days_observed_30d
future_available_days_30d
future_available_rate_30d
high_demand_proxy
```

Why?

- `high_demand_proxy` is the label.
- `future_*` columns are from the label window.
- `listing_id`, `cutoff_date`, and `dataset_version` are audit/entity fields, not predictive features.

You will intentionally use one future column in the **leaky run only** to show what leakage looks like. Your final model must be clean.

In [5]:
TARGET_COL = "high_demand_proxy"

FORBIDDEN_MODEL_COLUMNS = [
    "listing_id",
    "cutoff_date",
    "dataset_version",
    "future_calendar_days_observed_30d",
    "future_available_days_30d",
    "future_available_rate_30d",
    "high_demand_proxy",
]

# TODO: check that TARGET_COL exists.
assert TARGET_COL in feature_df.columns,f"Target column '{TARGET_COL}' not found in dataset."
# TODO: create y.
y=feature_df[TARGET_COL].astype(int)
# TODO: create clean feature list by excluding FORBIDDEN_MODEL_COLUMNS.
clean_feature_cols=[col for col in feature_df.columns if col not in FORBIDDEN_MODEL_COLUMNS]
# TODO: create X_clean.
X_clean=feature_df[clean_feature_cols].copy()

print("Target distribution:")
print(y.value_counts(normalize=True).sort_index())


print("Clean feature count:", len(clean_feature_cols))
print(clean_feature_cols)

Target distribution:
high_demand_proxy
0    0.237214
1    0.762786
Name: proportion, dtype: float64
Clean feature count: 26
['property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'listing_price', 'minimum_nights', 'maximum_nights', 'instant_bookable', 'is_superhost', 'host_listing_count', 'neighbourhood_name', 'total_reviews_before_cutoff', 'unique_reviewers_before_cutoff', 'avg_comment_len_before_cutoff', 'max_comment_len_before_cutoff', 'days_since_last_review', 'available_days_last_90d', 'available_rate_last_90d', 'avg_minimum_nights_calendar_last_90d', 'avg_maximum_nights_calendar_last_90d', 'available_days_last_30d', 'available_rate_last_30d', 'avg_minimum_nights_calendar_last_30d', 'avg_maximum_nights_calendar_last_30d']


## 4. Create one intentionally leaky feature set

This run is supposed to be wrong.

Create `X_leaky` by allowing `future_available_rate_30d` into the features.

The point is to show that a model can look excellent for the wrong reason. Log this run with:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
```

Do not select this run as your final model.

In [6]:
LEAKAGE_COLUMN = "future_available_rate_30d"

# TODO: create leaky_feature_cols.
# It should include the clean features plus LEAKAGE_COLUMN.
# It must still exclude the target itself.
leaky_feature_cols = clean_feature_cols +[LEAKAGE_COLUMN]
X_leaky= feature_df[leaky_feature_cols].copy()
print("Leaky feature count:", len(leaky_feature_cols))
print("Leakage column included:", LEAKAGE_COLUMN in leaky_feature_cols)

Leaky feature count: 27
Leakage column included: True


## 5. Train/test split

Use a stratified split.

Why stratified?

The target is not perfectly balanced, so the train and test sets should preserve the class ratio.

In [7]:
# TODO: split X_clean and y.
# Use test_size=0.20, random_state=42, stratify=y.
X_train,X_test,y_train,y_test=train_test_split(X_clean,y,test_size=0.20,random_state=42,stratify=y)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (8384, 26)
Test shape: (2096, 26)
Train target rate: 0.7627624045801527
Test target rate: 0.762881679389313


## 6. Build preprocessing

Use an sklearn `ColumnTransformer`.

Required preprocessing:

- numeric columns:
  - median imputation
  - standard scaling
- categorical columns:
  - most-frequent imputation
  - one-hot encoding

The logged model must be a full sklearn `Pipeline`, not just the estimator.

In [35]:
def make_one_hot_encoder():
    """Return OneHotEncoder compatible with multiple sklearn versions."""
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


# TODO: identify numeric_cols and categorical_cols from X_clean.
# Hint: numeric columns usually have dtype int/float.
# Everything else can be treated as categorical.
numeric_cols = X_clean.select_dtypes(include=["int","float","bool","boolean"]).columns.tolist()
categorical_cols = X_clean.select_dtypes(exclude=["int","float","bool","boolean"]).columns.tolist()


numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print(type(preprocessor))

Numeric columns: 23
Categorical columns: 3
<class 'sklearn.compose._column_transformer.ColumnTransformer'>


## 7. Evaluation helpers

Complete the evaluation helper.

Every run must log the same metric set:

```text
accuracy
precision
recall
f1
roc_auc
```

Use `zero_division=0` for precision/recall/f1.

In [10]:
def get_positive_scores(model, X):
    """Return positive-class scores for binary classifiers."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        raw = model.decision_function(X)
        return 1 / (1 + np.exp(-raw))
    return model.predict(X)


def evaluate_binary_classifier(model, X_test, y_test, threshold=0.5):
    """Evaluate a fitted binary classifier."""
    # TODO:
    # 1. get positive scores
    y_score = get_positive_scores(model,X_test)
    # 2. convert scores to predictions using threshold
    y_pred = (y_score>=threshold).astype(int)
    # 3. calculate accuracy, precision, recall, f1, roc_auc
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    }
    # 4. return metrics dict, y_pred, y_score
    return metrics, y_pred, y_score

## 8. Artifact helpers

Each serious run should save useful artifacts:

- confusion matrix image
- classification report JSON
- feature column list JSON
- dataset metadata snapshot JSON

Artifacts are important because MLflow should store more than scalar metrics.

In [11]:
ARTIFACT_DIR = Path("outputs/mlflow_artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


def save_run_artifacts(run_name, y_true, y_pred, feature_cols, metadata):
    """Save local artifact files for one run and return the run artifact directory."""
    # TODO:
    # 1. create a run-specific artifact folder
    run_dir = ARTIFACT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    # 2. save confusion_matrix.png
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix — {run_name}")
    plt.tight_layout()
    fig.savefig(run_dir / "confusion_matrix.png", dpi=100)
    plt.close(fig)

    # 3. save classification_report.json
    report = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    with open(run_dir / "classification_report.json", "w") as f:
        json.dump(report, f, indent=2)

    # 4. save feature_columns.json
    with open(run_dir / "feature_columns.json", "w") as f:
        json.dump({"feature_cols": list(feature_cols), "n_features": len(feature_cols)}, f, indent=2)

    # 5. save dataset_metadata_snapshot.json
    with open(run_dir / "dataset_metadata_snapshot.json", "w") as f:
        json.dump(metadata if metadata else {}, f, indent=2)
    return run_dir


## 9. MLflow run helper

Complete a helper that:

1. fits the pipeline,
2. evaluates it,
3. logs params,
4. logs metrics,
5. logs tags,
6. logs artifacts,
7. logs the full sklearn Pipeline model.

Use the same helper for all model versions. That is the point of experiment tracking.

In [21]:
from sklearn.base import clone
def run_mlflow_experiment(
    run_name,
    pipeline,
    X_train,
    X_test,
    y_train,
    y_test,
    feature_cols,
    model_params,
    tags,
    threshold=0.5,
):
    with mlflow.start_run(run_name=run_name) as run:
        # Fit the pipeline
        # pipeline.fit(X_train, y_train)
        fitted_pipeline = clone(pipeline)
        fitted_pipeline.fit(X_train, y_train)

        # Evaluate
        metrics, y_pred, y_score = evaluate_binary_classifier(fitted_pipeline, X_test, y_test, threshold=threshold)
 
        # TODO: implement this function.
        # Required MLflow calls:
        # - mlflow.start_run(run_name=run_name)
        # - mlflow.log_params(...)
        # - mlflow.log_metrics(...)
        # - mlflow.set_tags(...)
        # - mlflow.log_artifacts(...)
        # - mlflow.sklearn.log_model(...)
        mlflow.log_params(model_params)
        mlflow.log_metrics(metrics)
        mlflow.set_tags(tags)
        run_dir = save_run_artifacts(run_name, y_test, y_pred, feature_cols, metadata)
        mlflow.log_artifacts(str(run_dir))
        mlflow.sklearn.log_model(fitted_pipeline, artifact_path="model")

        print(f"Run: {run_name}")
        print(f"  Run ID: {run.info.run_id}")
        for k, v in metrics.items():
            print(f"  {k}: {v:.4f}")

        return run.info.run_id



## 10. Run 0 — intentionally leaky model

This run is wrong on purpose.

Use a real model, but include `future_available_rate_30d`.

Expected behavior: performance may look suspiciously strong.

Required tags:

```text
leakage_status = leaky
known_defect = uses future_available_rate_30d
model_family = logistic_regression
```

In [41]:
# TODO:
# 1. split X_leaky and y using the same stratified split settings
X_leaky_train, X_leaky_test, y_leaky_train, y_leaky_test = train_test_split(
    X_leaky, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# 2. build a LogisticRegression pipeline
leaky_numeric_cols = X_leaky.select_dtypes(include=["int","float","bool","boolean"]).columns.tolist()
leaky_categorical_cols = X_leaky.select_dtypes(exclude=["int","float","bool","boolean"]).columns.tolist()

leaky_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), leaky_numeric_cols),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]), categorical_cols),
    ],
    remainder="drop",
)

leaky_pipeline = Pipeline([
    ("preprocessor", leaky_preprocessor),
    ("classifier", LogisticRegression(max_iter=500, random_state=RANDOM_STATE)),
])
print(y_leaky_test.head())
# 3. log the run to MLflow
run_id_v0 = run_mlflow_experiment(
    run_name="v0_leaky_logistic_regression",
    pipeline=leaky_pipeline,
    X_train=X_leaky_train,
    X_test=X_leaky_test,
    y_train=y_leaky_train,
    y_test=y_leaky_test,
    feature_cols=leaky_feature_cols,
    model_params={"model": "LogisticRegression", "max_iter": 500},
    tags={
        "leakage_status": "leaky",
        "known_defect": "uses future_available_rate_30d",
        "model_family": "logistic_regression",
    },
)

3137    1
5589    1
9104    1
4272    1
9331    1
Name: high_demand_proxy, dtype: int64


2026/06/12 11:10:58 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/12 11:10:59 INFO mlflow.tracking._tracking_service.client: 🏃 View run v0_leaky_logistic_regression at: http://185.50.38.163:33014/#/experiments/29/runs/e20ae3d7962c43958ef7ee2244dc08f4.
2026/06/12 11:10:59 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


Run: v0_leaky_logistic_regression
  Run ID: e20ae3d7962c43958ef7ee2244dc08f4
  accuracy: 0.9995
  precision: 0.9994
  recall: 1.0000
  f1: 0.9997
  roc_auc: 1.0000


## 11. Run 1 — dummy baseline

Train a `DummyClassifier(strategy="most_frequent")`.

This tells you what a useless model can achieve.

If your real model barely beats this, your model is weak.

In [39]:
# TODO: build and log dummy baseline.


dummy_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)),
])

run_id_v1 = run_mlflow_experiment(
    run_name="v1_dummy_baseline",
    pipeline=dummy_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model": "DummyClassifier", "strategy": "most_frequent"},
    tags={
        "leakage_status": "clean",
        "model_family": "dummy",
    },
)


2026/06/12 11:05:25 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/12 11:05:27 INFO mlflow.tracking._tracking_service.client: 🏃 View run v1_dummy_baseline at: http://185.50.38.163:33014/#/experiments/29/runs/959113947f4f4cba9658ae5bd8cebcec.
2026/06/12 11:05:27 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


Run: v1_dummy_baseline
  Run ID: 959113947f4f4cba9658ae5bd8cebcec
  accuracy: 0.7629
  precision: 0.7629
  recall: 1.0000
  f1: 0.8655
  roc_auc: 0.5000


## 12. Run 2 — clean logistic regression

Train your first clean real model.

Use only `X_clean`.

Required tags:

```text
leakage_status = clean
model_family = logistic_regression
```

In [40]:
# TODO: build and log clean LogisticRegression.

clean_lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=500, random_state=RANDOM_STATE)),
])

run_id_v2 = run_mlflow_experiment(
    run_name="v2_clean_logistic_regression",
    pipeline=clean_lr_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model": "LogisticRegression", "max_iter": 500, "class_weight": "None"},
    tags={
        "leakage_status": "clean",
        "model_family": "logistic_regression",
    },
)

2026/06/12 11:09:57 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run: v2_clean_logistic_regression
  Run ID: ac293869acfe43e5aeae8f4c8366223d
  accuracy: 0.9699
  precision: 0.9717
  recall: 0.9894
  f1: 0.9805
  roc_auc: 0.9839


2026/06/12 11:09:58 INFO mlflow.tracking._tracking_service.client: 🏃 View run v2_clean_logistic_regression at: http://185.50.38.163:33014/#/experiments/29/runs/ac293869acfe43e5aeae8f4c8366223d.
2026/06/12 11:09:58 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


## 13. Run 3 — class-weighted logistic regression

Train logistic regression with:

```python
class_weight="balanced"
```

Compare precision and recall against the previous clean logistic model.

In [42]:
# TODO: build and log class-weighted LogisticRegression.


balanced_lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)),
])

run_id_v3 = run_mlflow_experiment(
    run_name="v3_balanced_logistic_regression",
    pipeline=balanced_lr_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={"model": "LogisticRegression", "max_iter": 500, "class_weight": "balanced"},
    tags={
        "leakage_status": "clean",
        "model_family": "logistic_regression",
    },
)


2026/06/12 11:11:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/12 11:11:20 INFO mlflow.tracking._tracking_service.client: 🏃 View run v3_balanced_logistic_regression at: http://185.50.38.163:33014/#/experiments/29/runs/60db6772360d4f5bb290c2758ff5b043.
2026/06/12 11:11:20 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


Run: v3_balanced_logistic_regression
  Run ID: 60db6772360d4f5bb290c2758ff5b043
  accuracy: 0.9742
  precision: 0.9783
  recall: 0.9881
  f1: 0.9832
  roc_auc: 0.9851


## 14. Run 4 — threshold tuning

Use a fitted probability model and test several decision thresholds.

Suggested thresholds:

```text
0.30, 0.40, 0.50, 0.60
```

You may log one run per threshold.

The goal is to see how precision/recall/f1 change when the threshold changes.

In [43]:
# TODO: log threshold-tuning runs.

threshold_base_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)),
])
threshold_base_pipeline.fit(X_train, y_train)

threshold_run_ids = {}
for thresh in [0.30, 0.40, 0.50, 0.60]:
    
    thresh_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=500, class_weight="balanced", random_state=RANDOM_STATE)),
    ])
    run_id = run_mlflow_experiment(
        run_name=f"v4_threshold_{str(thresh).replace('.', '_')}",
        pipeline=thresh_pipeline,
        X_train=X_train,
        X_test=X_test,
        y_train=y_train,
        y_test=y_test,
        feature_cols=clean_feature_cols,
        model_params={
            "model": "LogisticRegression",
            "max_iter": 500,
            "class_weight": "balanced",
        },
        tags={
            "leakage_status": "clean",
            "model_family": "logistic_regression",
            "experiment_type": "threshold_tuning",
        },
        threshold=thresh,
    )
    threshold_run_ids[thresh] = run_id

print("Threshold tuning complete.")


2026/06/12 11:11:50 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/12 11:11:51 INFO mlflow.tracking._tracking_service.client: 🏃 View run v4_threshold_0_3 at: http://185.50.38.163:33014/#/experiments/29/runs/91be5e7d0f5543b8a9d1f0670e52fb91.


Run: v4_threshold_0_3
  Run ID: 91be5e7d0f5543b8a9d1f0670e52fb91
  accuracy: 0.9719
  precision: 0.9741
  recall: 0.9894
  f1: 0.9817
  roc_auc: 0.9851


2026/06/12 11:11:51 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.
2026/06/12 11:11:55 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/12 11:11:56 INFO mlflow.tracking._tracking_service.client: 🏃 View run v4_threshold_0_4 at: http://185.50.38.163:33014/#/experiments/29/runs/bf18c29650e442698c23ad4ea2b5c7e8.
2026/06/12 11:11:56 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


Run: v4_threshold_0_4
  Run ID: bf18c29650e442698c23ad4ea2b5c7e8
  accuracy: 0.9733
  precision: 0.9759
  recall: 0.9894
  f1: 0.9826
  roc_auc: 0.9851


2026/06/12 11:12:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2026/06/12 11:12:02 INFO mlflow.tracking._tracking_service.client: 🏃 View run v4_threshold_0_5 at: http://185.50.38.163:33014/#/experiments/29/runs/fc946a161fcb4bf688f705fcfa552311.
2026/06/12 11:12:02 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


Run: v4_threshold_0_5
  Run ID: fc946a161fcb4bf688f705fcfa552311
  accuracy: 0.9742
  precision: 0.9783
  recall: 0.9881
  f1: 0.9832
  roc_auc: 0.9851


2026/06/12 11:12:06 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run: v4_threshold_0_6
  Run ID: cac90bdf831344589748320d9acd51d0
  accuracy: 0.9752
  precision: 0.9801
  recall: 0.9875
  f1: 0.9838
  roc_auc: 0.9851


2026/06/12 11:12:07 INFO mlflow.tracking._tracking_service.client: 🏃 View run v4_threshold_0_6 at: http://185.50.38.163:33014/#/experiments/29/runs/cac90bdf831344589748320d9acd51d0.
2026/06/12 11:12:07 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


Threshold tuning complete.


## 15. Run 5 — tree-based model

Train a `RandomForestClassifier`.

This compares a nonlinear model against logistic regression.

Log at least these parameters:

```text
n_estimators
max_depth
min_samples_leaf
class_weight
random_state
```

In [44]:
# TODO: build and log RandomForestClassifier.

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )),
])

run_id_v5 = run_mlflow_experiment(
    run_name="v5_random_forest",
    pipeline=rf_pipeline,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    feature_cols=clean_feature_cols,
    model_params={
        "model": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 10,
        "min_samples_leaf": 5,
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
    },
    tags={
        "leakage_status": "clean",
        "model_family": "random_forest",
    },
)


2026/06/12 11:12:30 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Run: v5_random_forest
  Run ID: c930c722d3674bf68497fc149c8d9a13
  accuracy: 0.9804
  precision: 0.9875
  recall: 0.9869
  f1: 0.9872
  roc_auc: 0.9923


2026/06/12 11:12:50 INFO mlflow.tracking._tracking_service.client: 🏃 View run v5_random_forest at: http://185.50.38.163:33014/#/experiments/29/runs/c930c722d3674bf68497fc149c8d9a13.
2026/06/12 11:12:50 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://185.50.38.163:33014/#/experiments/29.


## 16. Compare MLflow runs

Use `mlflow.search_runs` to retrieve your experiment runs.

Compare at least:

```text
run name
leakage status
model family
accuracy
precision
recall
f1
roc_auc
```

Do not select a leaky run as final candidate.

In [46]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

# TODO: retrieve MLflow runs for this experiment and create a comparison table.

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"],
)

comparison_cols = {
    "tags.mlflow.runName": "run_name",
    "tags.leakage_status": "leakage_status",
    "tags.model_family": "model_family",
    "metrics.accuracy": "accuracy",
    "metrics.precision": "precision",
    "metrics.recall": "recall",
    "metrics.f1": "f1",
    "metrics.roc_auc": "roc_auc",
    "run_id": "run_id",
}

available_cols = [c for c in comparison_cols if c in runs_df.columns]
comparison_df = runs_df[available_cols].rename(columns=comparison_cols)
comparison_df = comparison_df.sort_values("f1", ascending=False).reset_index(drop=True)

metric_cols = ["accuracy", "precision", "recall", "f1", "roc_auc"]
for col in metric_cols:
    if col in comparison_df.columns:
        comparison_df[col] = comparison_df[col].round(4)

print(comparison_df.to_string(index=False))
comparison_df


                       run_name leakage_status        model_family  accuracy  precision  recall     f1  roc_auc                           run_id
   v0_leaky_logistic_regression          leaky logistic_regression    0.9995     0.9994  1.0000 0.9997   1.0000 e20ae3d7962c43958ef7ee2244dc08f4
               v5_random_forest          clean       random_forest    0.9804     0.9875  0.9869 0.9872   0.9923 c930c722d3674bf68497fc149c8d9a13
               v4_threshold_0_6          clean logistic_regression    0.9752     0.9801  0.9875 0.9838   0.9851 cac90bdf831344589748320d9acd51d0
               v4_threshold_0_5          clean logistic_regression    0.9742     0.9783  0.9881 0.9832   0.9851 fc946a161fcb4bf688f705fcfa552311
v3_balanced_logistic_regression          clean logistic_regression    0.9742     0.9783  0.9881 0.9832   0.9851 60db6772360d4f5bb290c2758ff5b043
               v4_threshold_0_4          clean logistic_regression    0.9733     0.9759  0.9894 0.9826   0.9851 bf18c29650e442698c

,run_name,leakage_status,model_family,accuracy,precision,recall,f1,roc_auc,run_id
0,v0_leaky_logistic_regression,leaky,logistic_regression,0.9995,0.9994,1.0000,0.9997,1.0000,e20ae3d7962c43958ef7ee2244dc08f4
1,v5_random_forest,clean,random_forest,0.9804,0.9875,0.9869,0.9872,0.9923,c930c722d3674bf68497fc149c8d9a13
2,v4_threshold_0_6,clean,logistic_regression,0.9752,0.9801,0.9875,0.9838,0.9851,cac90bdf831344589748320d9acd51d0
3,v4_threshold_0_5,clean,logistic_regression,0.9742,0.9783,0.9881,0.9832,0.9851,fc946a161fcb4bf688f705fcfa552311
4,v3_balanced_logistic_regression,clean,logistic_regression,0.9742,0.9783,0.9881,0.9832,0.9851,60db6772360d4f5bb290c2758ff5b043
5,v4_threshold_0_4,clean,logistic_regression,0.9733,0.9759,0.9894,0.9826,0.9851,bf18c29650e442698c23ad4ea2b5c7e8
6,v4_threshold_0_3,clean,logistic_regression,0.9719,0.9741,0.9894,0.9817,0.9851,91be5e7d0f5543b8a9d1f0670e52fb91
7,v2_clean_logistic_regression,clean,logistic_regression,0.9699,0.9717,0.9894,0.9805,0.9839,ac293869acfe43e5aeae8f4c8366223d
8,v1_dummy_baseline,clean,dummy,0.7629,0.7629,1.0000,0.8655,0.5000,959113947f4f4cba9658ae5bd8cebcec


## 17. Select final candidate

Pick the best **clean** run.

Do not choose the leaky run.

Selection should be based on:

- f1
- roc_auc
- precision/recall tradeoff
- no leakage
- full preprocessing Pipeline logged

Write a short explanation.

In [47]:
# TODO: set BEST_RUN_ID to the selected clean run ID.
clean_runs = comparison_df[comparison_df["leakage_status"] == "clean"].copy()
best_clean_run = clean_runs.sort_values("f1", ascending=False).iloc[0]
BEST_RUN_ID = best_clean_run["run_id"]

print("Best clean run name:", best_clean_run["run_name"])
print("F1:", best_clean_run["f1"])
print("ROC-AUC:", best_clean_run["roc_auc"])

client.set_tag(BEST_RUN_ID, "selected_for_serving", "true")
client.set_tag(BEST_RUN_ID, "production_candidate", "true")

print("Selected best run:", BEST_RUN_ID)

Best clean run name: v5_random_forest
F1: 0.9872
ROC-AUC: 0.9923
Selected best run: c930c722d3674bf68497fc149c8d9a13


## Final explanation

Write 3–6 sentences:

- Which run did you select?
- Why did you select it?
- Why did you reject the leaky run?
- What would you try next?

In [48]:
# TODO: replace this text.
final_explanation = """
I selected v5_random_forest as the final model. It achieved the best performance among the clean models.
I rejected v0_leaky_logistic_regression because information from the target leaked into the training process, causing the model to learn patterns it would not have access to in real-world predictions
For next steps,tune the Random Forest hyperparameters (number of trees, depth, minimum samples per split).I would also compare the Random Forest against more advanced models such as Gradient Boosting or XGBoost.
"""

print(final_explanation)


I selected v5_random_forest as the final model. It achieved the best performance among the clean models.
I rejected v0_leaky_logistic_regression because information from the target leaked into the training process, causing the model to learn patterns it would not have access to in real-world predictions
For next steps,tune the Random Forest hyperparameters (number of trees, depth, minimum samples per split).I would also compare the Random Forest against more advanced models such as Gradient Boosting or XGBoost.

